# Experiment B — rotationally tied learned features

**Frozen U-Net → anchored projected gradient descent → trainable structured energy → 8-connected nuclear instance evaluation.**

The MM and original separator/boundary kernel shapes remain fixed. Their positive strengths are learnable and capped at 1.5, starting at 0.5 each. There are 2–4 additional **canonical trainable two-channel filters**, each expanded to a differentiably rotated bank (default 16 orientations over 360°); orientation responses use max → ReLU → squared energy reward. New strengths start at **0.05** and are capped at 0.3.

All parameters are shared across refinement steps. Defaults here use SGD at **lr=1.0** (as in your recent test). The original weighted CE + soft Dice loss, L2 anchor, stochastic corruption, warmups and full validation are unchanged.

**Checkpoint note:** Rotated-feature checkpoints are a different architecture from the earlier independent-feature experiment; use a fresh output folder. Previous best/latest checkpoints cannot be loaded strictly into this model.


## 1. Install dependencies and upload the experiment package

In [ ]:
%pip -q install psfmodels scipy scikit-image numba matplotlib pandas

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
uploaded = files.upload()  # Upload experiment_b_rotated_bundle.zip
from pathlib import Path
import zipfile, sys
bundle = next((Path(s) for s in uploaded if s.endswith('.zip')), None)
if bundle is None: raise RuntimeError('Upload experiment_b_rotated_bundle.zip')
with zipfile.ZipFile(bundle) as zf: zf.extractall('/content')
SRC = Path('/content/experiment_b_rotated')
assert (SRC / 'train_experiment_b.py').exists()
sys.path.insert(0, str(SRC))
print('Code:', SRC)

## 2. Choose the pretrained U-Net checkpoint

The loader strictly checks U-Net state-dict compatibility. If the checkpoint directory contains multiple files, select a specific `.pt` file.

In [ ]:
from unet_inference import locate_checkpoint
CHECKPOINT = Path('/content/drive/MyDrive/Colab Notebooks/proj_26/checkpoints_unet_baseline_v1_boosted_ce')
if CHECKPOINT.is_dir():
    candidates = sorted(x for x in CHECKPOINT.rglob('*') if x.suffix in ('.pt','.pth','.ckpt','.bin'))
    print('Available checkpoints:')
    for c in candidates: print(' ', c)
    if len(candidates) > 1:
        print('Set CHECKPOINT to the exact file you want, e.g. CHECKPOINT = CHECKPOINT / "latest.pt"')
print('Selected:', CHECKPOINT)

## 3. Configuration

**Pilot:** 2 training volumes, 1 validation volume, ~10 updates, short horizons. **Full:** raise volumes/updates/validation slices and turn on the long warmups. Training data and validation data have nonoverlapping seed ranges. This does not certify independence from your pretrained U-Net's training set.

The existing fixed energy geometry and coefficients below match your last provided configuration. The first energy step uses `step_size=0.02`, `anchor_mu=0.4`; projected descent uses your original reduced-coordinate Euclidean projection.

**SGD note:** This is vanilla SGD (`momentum=0`), with LR `1e-3` and weight decay `1e-5`. Because this is higher-order differentiation, the gradients may be small; the CSV logs let you check that before increasing the learning rate.

In [ ]:
import torch
from train_experiment_b import TrainBConfig, train
cfg = TrainBConfig(
    checkpoint=str(CHECKPOINT),
    output_dir='/content/drive/MyDrive/Colab Notebooks/proj_26/experiment_b_rotated_results',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=2400, train_seed=200000, val_seed=300000,
    train_volumes=8, val_volumes=3, N=192, z_slices=16,
    z_ratio=2, n_seeds=45, class_probs=(2/5,1/5,2/5),
    train_domain='a', val_domain='a',
    n_extra=3, extra_filter_size=13, extra_orientations=16,
    strength_cap=1.5, extra_strength_cap=0.3,
    kernel_raw_cap=2., initial_extra_strength=.05,
    anchor_mu=.4, step_size=.02,
    updates=250, learning_rate=1.0, momentum=0., weight_decay=1e-5,
    batch_size=2, warmups=(0,20,80,160,300), differentiable_steps=4,
    source_mode='mixed', prob_unet_in_mixed=.5,  # 'unet' / 'corrupted' also work
    gt_blur_range=(0.,1.5), gt_spectral_range=(.4,1.2),
    restart_probability=.5, restart_blur_range=(0.,.8), restart_spectral_range=(0.,.5),
    eval_every=25, val_horizons=(0,50,150,300,400),
    validation_max_slices=8,
    energy=dict(
        lam=.2, epsilon=.5, well_cubic_weight=1.,
        barrier_weight=.5, barrier_width=5, nuclear_band_width=2,
        barrier_length=13, barrier_longitudinal_sigma=3.5,
        barrier_orientations=8,
        boundary_weight=.5, boundary_middle_width=5,
        boundary_outer_width=2, boundary_orientations=16,
        learn_strengths=False, learn_residuals=False,
    ),
)
print('Device:', cfg.device)

## 4. Optional quick pilot (recommended first)

This uses a **separate pilot directory**, leaving the full configuration unchanged.

In [ ]:
from dataclasses import replace
pilot = replace(cfg, train_volumes=1, val_volumes=1, updates=3,
                warmups=(0,2), val_horizons=(0,2,5),
                validation_max_slices=2, eval_every=2,
                output_dir=str(Path(cfg.output_dir) / 'pilot'))
# Uncomment to check checkpoint compatibility, data generation and gradients:
# pilot_model, pilot_history, pilot_validation = train(pilot)

## 5. Train experiment B

Captures training histories and checkpoints to Google Drive. This is computationally expensive: increase training/validation sizes only after the pilot succeeds.

In [ ]:
model, history, validation_history = train(cfg)

## 6. Plot training dynamics and held-out instance metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
out=Path(cfg.output_dir)
tr=pd.read_csv(out/'train_history.csv')
va=pd.read_csv(out/'validation_history.csv')
fig,axes=plt.subplots(1,3,figsize=(16,4))
axes[0].plot(tr['update'],tr['loss']); axes[0].set(title='Training objective',xlabel='Update',ylabel='Loss')
for k in ['separator_strength','boundary_strength']+[f'extra_strength_{i}' for i in range(cfg.n_extra)]:
    axes[1].plot(tr['update'],tr[k],label=k)
axes[1].legend(fontsize=7); axes[1].set(title='Learned filter strengths',xlabel='Update')
for t in sorted(va['iteration'].unique()):
    d=va[va['iteration']==t]
    axes[2].plot(d['update'],d['micro_instance_f1'],label=f'{t} iterations')
axes[2].legend(fontsize=8); axes[2].set(title='Held-out instance micro F1',xlabel='Update',ylabel='F1')
fig.tight_layout(); fig.savefig(out/'training_curves.png',dpi=150); plt.show()
print(va.groupby('iteration').tail(1).to_string(index=False))

## 7. Evaluate Domain B/C without retraining (optional)

The simplest method is to call `prepare_cache` and `evaluate` using the saved best checkpoint. The U-Net remains frozen and the learned energy is unchanged. Use a **new seed range** for final test domains and do not use their F1 values to choose hyperparameters.

In [ ]:
# Optional example; uncomment only after training.
# from train_experiment_b import prepare_cache, evaluate
# from unet_inference import load_unet
# net, _ = load_unet(str(CHECKPOINT), cfg.device)
# test_cache = prepare_cache(net, cfg, seed=400000, n_volumes=10, domain='b')
# from learned_energy import StructuredEnergy
# from mm_fixed_compatible import Config
# best = torch.load(Path(cfg.output_dir)/'best.pt',map_location=cfg.device,weights_only=False)
# saved_tc = best['training_config']
# learned = StructuredEnergy(Config(**best['model_config']),cfg.n_extra,cfg.extra_filter_size,
#      cfg.strength_cap,cfg.extra_strength_cap,cfg.kernel_raw_cap,cfg.initial_extra_strength,cfg.seed).to(cfg.device)
# learned.load_state_dict(best['model_state_dict'])
# per_slice, summary = evaluate(learned,test_cache,cfg,max_slices=None)
# print(summary)

## Design notes

- `learned_energy.py` implements both original fixed filter families with bounded trainable strengths and a small bank of new rotation-tied learned kernels.
- Each new canonical filter is rotated with the original SciPy bilinear convention, spatially zero-centered and positive-sum normalized **per orientation**, then aggregated with orientation max and ReLU.
- `extra_orientations=16` is sampled over 360°; `n_extra=3` canonical filters, not 48 independent kernels.
- The learned energy and fixed parameters are shared across all gradient-descent iterations. The full three-class anchor term remains separate from the energy.
- The 400-step validation can take substantially longer than with the original unrotated learned kernels. Test pilot settings first.
- `best.pt` / `latest.pt` from the **previous** independent-feature experiment are not compatible; start a new output directory.
- Learnable filter strengths are initialized to 0.05 and capped at 0.3.
- The frozen U-Net, data generator and supervised loss remain unchanged.


## 8. Visualize raw U-Net vs initial and learned energy

This comparison uses the same original probability input, anchor and step size. The learned energy loads the new checkpoint including rotated features; the initial energy uses the original fixed separator and boundary filters at 0.5 each. Choose domain, volume, slice, and checkpoint below.


In [ ]:
# Paste this entire cell after Experiment B's training cells.
# Assumes cfg and CHECKPOINT are defined by the original notebook.
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from synthetic_fluor_stains_three_classes_weighted import SyntheticFluorStains
from unet_inference import load_unet, predict_volume
from mm_fixed_compatible import Config, Energy, target_from_stain_labels
from learned_energy import StructuredEnergy, anchored_rollout
from instance_metrics import instance_stats, dice_by_class

# -------- Choose what to inspect --------
VIS_DOMAIN = 'b'             # 'a', 'b', or 'c'
VIS_SEED = 400000           # use held-out seed for domain comparisons
VIS_VOLUME = 0             # synthetic dataset index, zero-based
VIS_SLICE = 8              # 0 ... cfg.z_slices-1
VIS_CHECKPOINT = 'best.pt'  # 'best.pt' or 'latest.pt'
VIS_ITERATIONS = (0, 100, 200, 300, 400)
VIS_SAVE_PNG = False
VIS_DEVICE = cfg.device

# -------- Load learned energy and reconstruct INITIAL fixed energy --------
state_path = Path(cfg.output_dir) / VIS_CHECKPOINT
if not state_path.is_file():
    raise FileNotFoundError(f'Missing {state_path}; train Experiment B or adjust VIS_CHECKPOINT')
state = torch.load(state_path, map_location='cpu', weights_only=False)
saved_tc = state['training_config']
saved_energy_cfg = state['model_config']
# Use parameters from the checkpoint, not possibly modified notebook cfg values.
energy_cfg = Config(**saved_energy_cfg)
learned_energy = StructuredEnergy(
    energy_cfg, n_extra=saved_tc['n_extra'],
    filter_size=saved_tc['extra_filter_size'],
    strength_cap=saved_tc['strength_cap'],
    extra_strength_cap=saved_tc['extra_strength_cap'],
    kernel_raw_cap=saved_tc['kernel_raw_cap'],
    initial_extra_strength=saved_tc['initial_extra_strength'],
    seed=saved_tc['seed'],
    extra_orientations=saved_tc.get('extra_orientations', 16),
).to(VIS_DEVICE)
learned_energy.load_state_dict(state['model_state_dict'], strict=True)
learned_energy.eval().requires_grad_(False)

# Original fixed energy: no extra filters, with strengths 0.5 and 0.5.
# These values are taken from the saved energy config, which was initialized at 0.5.
fixed_energy = Energy(Config(**saved_energy_cfg)).to(VIS_DEVICE)
fixed_energy.eval().requires_grad_(False)
print('Learned strengths:', learned_energy.diagnostics())
print('Original strengths:', saved_energy_cfg['barrier_weight'], saved_energy_cfg['boundary_weight'])
print('Loaded update:', state.get('update'))

# -------- Load a SINGLE synthetic volume and predict its selected slice --------
dataset = SyntheticFluorStains(
    samples_per_epoch=max(VIS_VOLUME + 1, 1), seed=VIS_SEED,
    N=saved_tc['N'], z_slices=saved_tc['z_slices'],
    z_ratio=saved_tc['z_ratio'], n_seeds=saved_tc['n_seeds'],
    class_probs=tuple(saved_tc['class_probs']), domain=VIS_DOMAIN,
)
images, labels = dataset[VIS_VOLUME]
assert 0 <= VIS_SLICE < images.shape[0]
unet, unet_path = load_unet(saved_tc['checkpoint'], VIS_DEVICE)
raw_volume = predict_volume(unet, images, VIS_DEVICE,
                            batch_size=saved_tc['inference_batch_size'])
del unet
raw = raw_volume[VIS_SLICE:VIS_SLICE+1].to(VIS_DEVICE).detach()
target = target_from_stain_labels(torch.as_tensor(labels))[VIS_SLICE].cpu().numpy()

# -------- Track both trajectories from EXACTLY the same original softmax --------
iterations = sorted(set(VIS_ITERATIONS) | {0})
mu = float(saved_tc['anchor_mu'])
tau = float(saved_tc['step_size'])

def snapshots(energy):
    result = {0: raw.detach().cpu().squeeze(0).clone()}
    current = raw
    last = 0
    for t in iterations[1:]:
        current = anchored_rollout(energy, current, raw,
                                   t - last, mu, tau, differentiable=False)
        result[t] = current.detach().cpu().squeeze(0).clone()
        last = t
    return result

fixed_history = snapshots(fixed_energy)
learned_history = snapshots(learned_energy)
raw_cpu = raw.detach().cpu().squeeze(0)

# -------- Compare classes and hard argmax labels at selected iteration --------
SHOW_STEP = max(iterations)    # change to any value from VIS_ITERATIONS
class_names = ('Background', 'Nuclear', 'Nonnuclear')
views = [
    ('Ground truth', None),
    ('Raw U-Net', raw_cpu.numpy()),
    (f'Initial energy ({SHOW_STEP})', fixed_history[SHOW_STEP].numpy()),
    (f'Learned energy ({SHOW_STEP})', learned_history[SHOW_STEP].numpy()),
]
semantic_cmap = ListedColormap(['#202934', '#72ace6', '#eeae60'])
semantic_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3)
fig, axes = plt.subplots(4, 4, figsize=(17, 15), constrained_layout=True)
for col, (name, probs) in enumerate(views):
    hard = target if probs is None else probs.argmax(axis=0)
    for row in range(4):
        ax = axes[row, col]
        if row == 0:
            ax.imshow(np.ma.masked_where(hard == -100, hard), cmap=semantic_cmap,
                      norm=semantic_norm, interpolation='nearest')
        else:
            k = row - 1
            channel = (target == k).astype(float) if probs is None else probs[k]
            ax.imshow(channel, cmap='viridis', vmin=0, vmax=1, interpolation='nearest')
        if row == 0:
            if probs is None:
                subtitle = 'Reference instances'
            else:
                st = instance_stats(hard, target)
                subtitle = f"F1={st['instance_f1']:.3f}  merges={st['merge_count']}  splits={st['split_count']}"
            ax.set_title(name + '\n' + subtitle, fontsize=11)
        if col == 0:
            ax.set_ylabel('Argmax labels' if row == 0 else class_names[row - 1], fontsize=12)
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f'Domain {VIS_DOMAIN.upper()}, seed {VIS_SEED}, volume {VIS_VOLUME}, Z={VIS_SLICE}', fontsize=15)
plt.show()

# -------- Optional: visualize how instance performance evolves --------
fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for name, history in [('Initial fixed energy', fixed_history), ('Learned energy', learned_history)]:
    scores = [instance_stats(history[t].argmax(0).numpy(), target) for t in iterations]
    axes2[0].plot(iterations, [s['instance_f1'] for s in scores], marker='o', label=name)
    axes2[1].plot(iterations, [s['merge_count'] for s in scores], marker='o', label=name + ' merges')
    axes2[1].plot(iterations, [s['split_count'] for s in scores], marker='s', linestyle='--', label=name + ' splits')
axes2[0].set(title='Instance F1 on selected slice', xlabel='Iteration', ylabel='F1')
axes2[1].set(title='Instance merge / split counts', xlabel='Iteration', ylabel='Count')
for ax in axes2: ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.show()

if VIS_SAVE_PNG:
    dest = Path(cfg.output_dir) / 'visualizations'
    dest.mkdir(parents=True, exist_ok=True)
    tag = f'{VIS_DOMAIN}_seed{VIS_SEED}_v{VIS_VOLUME}_z{VIS_SLICE}_{VIS_CHECKPOINT.split(".")[0]}'
    fig.savefig(dest / f'{tag}_channels.png', dpi=170, bbox_inches='tight')
    fig2.savefig(dest / f'{tag}_trajectory.png', dpi=170, bbox_inches='tight')
    print('Saved figures in', dest)
